# TensorFlow Serving

A comprehensive guide to TensorFlow Serving for AI/ML workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

TensorFlow Serving is a production-grade system for serving TensorFlow models (and, via SavedModel, also some non-TF models) at scale. It is part of the broader TensorFlow Extended (TFX) ecosystem and is widely used to deploy deep learning models behind low-latency, high-throughput APIs.

This notebook gives you a practical, end‑to‑end overview of TensorFlow Serving in the context of real AI/ML workloads.

### What is it?

At a high level, **TensorFlow Serving** is:

- A **model server** process (typically run as a container) that loads one or more exported TensorFlow `SavedModel`s from disk.
- A **gRPC/HTTP API layer** that exposes prediction endpoints (`Predict`, `Classify`, `Regress`) for online inference.
- A **model lifecycle manager** that watches file system paths for new model versions and performs **hot model reloads** without downtime.

You usually interact with TensorFlow Serving from an external client (Python, Java, Go, curl) rather than by importing it as a library in your training code.

### Why use it?

Key benefits of using TensorFlow Serving:

- **Standardized serving for TF models**  
  Uses the `SavedModel` format, so your training code and serving stack share a common artifact.
- **High performance C++ server**  
  Built in C++ with efficient batching and threading, designed for low‑latency, high‑throughput inference.
- **Versioned model management**  
  Supports multiple model versions and automatic roll‑forward/rollback via simple directory operations.
- **gRPC and REST APIs**  
  Works well with polyglot clients and can be fronted by API gateways, load balancers, or service meshes.
- **First‑class GPU support**  
  Works with CUDA‑enabled builds to serve models on GPUs.

### When to use it?

TensorFlow Serving is particularly useful when:

- **Your models are trained in TensorFlow/Keras** and exported as `SavedModel`s.
- You need a **stable, well‑understood serving stack** operated by infra/SRE teams (on‑prem or cloud).
- You want **simple deployment workflows** based on model folders and container images (no custom app code).
- You are building **online prediction services** with tight latency SLAs.

You might reach for alternatives (e.g., vLLM, Triton Inference Server, FastAPI+custom code) when:

- You primarily serve non‑TF models (e.g., PyTorch without ONNX/SavedModel conversion).
- You need highly specialized batching or multi‑framework serving beyond what TF Serving offers.
- You want a more customizable Python-serving stack with business logic tightly integrated into the server.

## Key Features

### Core capabilities of TensorFlow Serving

| Feature | Description | Benefit |
|--------|-------------|---------|
| **SavedModel-based serving** | Loads models directly from TensorFlow `SavedModel` directories. | Simple, standardized deployment flow from training to serving. |
| **Versioned models** | Watches a model base path and automatically loads the latest version (e.g., `models/my_model/0001`, `0002`, …). | Enables safe rollouts, rollbacks, and A/B testing by changing symlinks or directories. |
| **Multi-model serving** | Can serve multiple models (and model versions) from a single server instance. | Reduces operational overhead; centralizes inference traffic. |
| **High-performance C++ core** | Server is implemented in C++ with efficient threading and batching. | Low latency and high throughput under heavy production load. |
| **Batching support** | Optional request batching at the server level. | Improved GPU/CPU utilization and cost efficiency. |
| **gRPC + REST APIs** | Exposes both gRPC and HTTP/JSON prediction APIs. | Flexible integration with polyglot clients and API gateways. |
| **Model warmup** | Supports warmup requests via a special file in the model directory. | Avoids cold-start latency for the first real requests. |
| **GPU support** | CUDA-enabled builds can serve models on NVIDIA GPUs. | Leverages hardware acceleration for deep learning workloads. |
| **Configuration via flags or config file** | Behavior controlled with command-line flags or a `models.config` file. | Repeatable, declarative configuration for infra teams. |

### Common MLOps-friendly properties

- **Stateless server**: All state is in the model files and configuration; servers are easy to scale horizontally.
- **Cloud-agnostic**: Packaged as a container image and deployed on VMs, Kubernetes, or managed services.
- **Rich ecosystem**: Works well with TFX pipelines, Kubernetes operators, and standard monitoring stacks (Prometheus, Grafana, etc.).

## Architecture Overview

At a high level, TensorFlow Serving sits between your **model artifacts** and your **client applications**.

```text
               ┌───────────────────────────────┐
               │       Client Apps             │
               │  • Web / mobile frontends     │
               │  • Backend services / APIs    │
               │  • Batch jobs / ETL           │
               └───────────────┬───────────────┘
                               │  HTTP / gRPC
                               ▼
                    ┌───────────────────────┐
                    │  TensorFlow Serving   │
                    │  (tensorflow_model_   │
                    │       server)         │
                    └─────────┬─────────────┘
                              │
                              ▼
                 ┌──────────────────────────┐
                 │   Model Repository       │
                 │  /models/my_model/      │
                 │    ├── 0001/            │
                 │    │    └── SavedModel  │
                 │    └── 0002/            │
                 │         └── SavedModel  │
                 └──────────────────────────┘
```

### Components

1. **Model repository (file system or volume)**
   - Directory tree containing one or more models and their versions.  
   - Example layout: `models/my_model/0001/`, `models/my_model/0002/`.

2. **TensorFlow Model Server (`tensorflow_model_server`)**
   - Long‑running C++ process that:
     - Loads `SavedModel` directories into memory.
     - Exposes prediction APIs over gRPC and REST.
     - Watches the model base path for new versions.

3. **Model configuration**
   - Either **implicit** (one model per base path) or **explicit** via a `models.config` file that declares models, versions, and policies.

4. **Clients**
   - Any service that can speak HTTP/JSON or gRPC.  
   - Typical patterns:
     - Backend service making per‑request predictions.
     - Batch job sending large prediction batches.
     - Online feature stores or stream processors calling the server.

5. **Optional: Load balancer / service mesh**
   - In Kubernetes, you typically run **multiple replicas** of the TF Serving container behind a `Service` or Ingress.
   - A service mesh (Istio, Linkerd, etc.) can add routing, retries, and mTLS.

6. **Observability stack**
   - Export metrics to Prometheus / OpenTelemetry.
   - Ship logs to centralized logging (e.g., Elasticsearch, Cloud Logging).

This architecture is intentionally **simple and composable**: TF Serving focuses on efficient model execution, and you integrate it into your broader platform for routing, auth, and observability.

## Installation

TensorFlow Serving is typically run as a **separate server process**, most often inside a container. You generally do **not** `pip install` it into the same environment as your training code.

### 1. Docker (recommended for most users)

The preferred way to run TensorFlow Serving is via the official Docker images:

```bash
# CPU-only image
docker pull tensorflow/serving

# GPU-enabled image (requires NVIDIA drivers + nvidia-container-runtime)
docker pull tensorflow/serving:latest-gpu
```

Run a basic CPU server (we will add a model later):

```bash
docker run -p 8500:8500 -p 8501:8501 \
  --name tf-serving \
  tensorflow/serving
```

- Port **8500**: gRPC API
- Port **8501**: REST API

### 2. Native packages (Linux only, less common now)

On some Linux distributions you can install `tensorflow_model_server` from a package repository (e.g., Google’s apt repo). This is useful for VM-based deployments without containers.

Conceptually, you end up with a `tensorflow_model_server` binary you run as a system service.

### 3. Local development in notebooks (this repo)

Inside a Jupyter environment you usually:

- Export a **SavedModel** from Python.
- Use **Docker** (running locally or remotely) to serve that model.
- Call the server from notebook cells using HTTP/gRPC clients.

We’ll follow this pattern in the **Basic Usage** section: train/export a simple model, start a TF Serving container that mounts the model, then send prediction requests from Python.

> **Note:** The rest of this notebook assumes you can run Docker commands from your environment (local machine, dev VM, or remote host).

In [ ]:
# Optional: install client libraries in this environment
# (TensorFlow for exporting SavedModels, requests for calling the REST API)

# Uncomment if needed in Colab or a fresh environment:
# !pip install "tensorflow>=2.10" requests

## Basic Usage

In this section we'll walk through a minimal end‑to‑end flow:

1. Export a simple TensorFlow model as a `SavedModel` in a **versioned directory**.
2. Start a TensorFlow Serving Docker container that mounts that model.
3. Send a REST request from Python to get a prediction.

This pattern scales from toy examples to production: training jobs export `SavedModel`s, and TensorFlow Serving instances watch a model directory and expose prediction APIs.

In [ ]:
# Basic end-to-end example for TensorFlow Serving

import os
import pathlib
import json

import numpy as np
import requests
import tensorflow as tf

# Paths
MODEL_NAME = "demo_mlp"
MODEL_BASE_PATH = pathlib.Path("models") / MODEL_NAME
VERSION = "0001"  # TensorFlow Serving expects numeric version subdirs
EXPORT_PATH = MODEL_BASE_PATH / VERSION

# 1. Define and export a simple Keras model as a SavedModel

# Simple dense model for demonstration (e.g., 4-dim input -> 3 classes)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(3, activation="softmax"),
])

model.compile(optimizer="adam", loss="categorical_crossentropy")

# Normally you would train the model here.
# For brevity, we'll skip training and just export the randomly initialized model.
EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
tf.saved_model.save(model, str(EXPORT_PATH))

print(f"SavedModel exported to: {EXPORT_PATH}")

# 2. Start TensorFlow Serving (run this in a terminal, not inside the notebook):
#
# docker run -p 8500:8500 -p 8501:8501 \
#   --mount type=bind,source=$(pwd)/models,target=/models \
#   -e MODEL_NAME={MODEL_NAME} \
#   tensorflow/serving \
#   --model_name={MODEL_NAME} \
#   --model_base_path=/models/{MODEL_NAME}
#
# This will expose:
# - gRPC on localhost:8500
# - REST  on localhost:8501

# 3. Send a REST prediction request from Python

# Create a dummy batch of inputs (batch_size=2, feature_dim=4)
inputs = np.array(
    [
        [5.1, 3.5, 1.4, 0.2],  # Example 1 (e.g., Iris features)
        [6.7, 3.1, 4.7, 1.5],  # Example 2
    ],
    dtype=np.float32,
)

payload = {"instances": inputs.tolist()}

try:
    resp = requests.post(
        f"http://localhost:8501/v1/models/{MODEL_NAME}:predict",
        json=payload,
        timeout=5,
    )
    resp.raise_for_status()
    predictions = resp.json()["predictions"]
    print("Predictions:")
    print(json.dumps(predictions, indent=2))
except Exception as e:
    print("Request to TensorFlow Serving failed. Make sure the Docker container is running.")
    print(f"Error: {e}")

## Advanced Features

TensorFlow Serving has several powerful capabilities that matter in real production setups.

### 1. Multi-model and multi-version serving

- You can serve **multiple models** from a single server by providing a `models.config` file.
- Each model can have **multiple versions** (e.g., `0001`, `0002`), and the server can:
  - Load the **latest N versions**.
  - Mark one version as "default" while still allowing requests to specific versions.

This enables patterns like **canary releases** and **A/B testing** using traffic routing at the client or load balancer layer.

### 2. Model version policies

Using `--model_version_policy` or the config file, you can control which versions are loaded:

- `latest(N)`: Only keep the latest N versions in memory.
- `specific(versions: [...])`: Pin to specific versions.
- `all()`: Load all available versions.

This is important for **memory management** and **safe rollbacks**.

### 3. Request batching

TensorFlow Serving supports server-side batching to improve throughput:

- Configure a **batching parameters file** and pass `--enable_batching=true`.
- The server will collect small incoming requests into larger batches before running them through the model.

This is particularly useful for **GPU-backed** deployments, where large batches use hardware more efficiently.

### 4. Custom signatures

SavedModels can export multiple **signatures** (e.g., `serving_default`, `predict`, `masked_language_model`). TensorFlow Serving lets you:

- Select a specific signature when making gRPC requests.
- Expose different behaviors (e.g., raw logits vs. post-processed predictions) under different endpoints.

### 5. Model warmup

To avoid cold-start latency, TF Serving can execute a **warmup request** on startup:

- Place a special warmup file under `assets.extra` in the SavedModel directory.
- The server replays these requests at startup so that kernels, caches, and graphs are ready when real traffic arrives.

### 6. Custom platforms and non-TF models

Although it is optimized for TensorFlow `SavedModel`s, TF Serving can be extended with **custom platforms** for other model types (e.g., via ONNX or custom C++ code). In practice, for non-TF models many teams now prefer more general serving stacks, but it’s useful to know the extension points exist.

In [ ]:
# Advanced feature example: multi-model, multi-version configuration

# In real deployments you often manage multiple models and versions.
# TensorFlow Serving can be configured with a `models.config` file like this:

models_config = """
model_config_list: {
  config: {
    name: "recommender",
    base_path: "/models/recommender",
    model_platform: "tensorflow",
    model_version_policy: {
      latest { num_versions: 2 }
    }
  }
  config: {
    name: "fraud_detector",
    base_path: "/models/fraud_detector",
    model_platform: "tensorflow",
    model_version_policy: {
      specific { versions: 1 versions: 3 }
    }
  }
}
"""

print(models_config)

# You would then start TensorFlow Serving with:
#
# docker run -p 8500:8500 -p 8501:8501 \
#   --mount type=bind,source=$(pwd)/models,target=/models \
#   --mount type=bind,source=$(pwd)/models.config,target=/models/models.config \
#   tensorflow/serving \
#   --model_config_file=/models/models.config
#
# This single server instance now serves multiple models and multiple versions.

## Use Cases

### 1. Online prediction microservice for TensorFlow models

- **Scenario**: You train Keras / TF models (e.g., recommendation, ranking, fraud detection) and need a low‑latency prediction API.
- **Pattern**:
  - Training jobs export `SavedModel`s to a shared model repository (object store or NFS).
  - TensorFlow Serving instances run behind a load balancer, watching that repository.
  - Application backends call TF Serving via gRPC or REST.
- **Why TF Serving fits**: Minimal custom code, strong performance, standardized artifact format.

### 2. Multi-model serving (feature store + multiple tasks)

- **Scenario**: A single platform needs to serve many small/medium TF models (e.g., per‑tenant models, per‑feature models).
- **Pattern**:
  - Use `models.config` to declare multiple models in one TF Serving instance.
  - Route requests to different models using different endpoints or metadata.
- **Benefits**:
  - Operational simplicity (fewer processes to manage).
  - Easier cross‑team standardization on one serving stack.

### 3. A/B testing and canary rollouts of model versions

- **Scenario**: You want to test a new model version on a small slice of traffic before full rollout.
- **Pattern**:
  - Export new versions as `0002`, `0003`, etc.
  - Configure TF Serving to load multiple versions.
  - Use the client or API gateway to send a small percentage of traffic to the new version.
- **Benefits**:
  - Safe, controlled experimentation with quick rollback (just switch which version you call).

### 4. Batch scoring via TF Serving

- **Scenario**: Nightly or hourly batch jobs need to score millions of records.
- **Pattern**:
  - A batch job (e.g., Spark, Beam, or Python script) sends batched requests to TF Serving.
  - Optionally enable server‑side batching for better throughput.
- **Benefits**:
  - Reuse the same serving stack for both online and batch workloads.
  - Centralized model rollback and monitoring.

### 5. Integration with TFX pipelines

- **Scenario**: You use TensorFlow Extended (TFX) for training and evaluation.
- **Pattern**:
  - TFX pipelines export validated models to a serving bucket/path.
  - A deployment step updates TF Serving’s model path or version.
- **Benefits**:
  - Clear separation of concerns: TFX handles training/validation; TF Serving handles online inference.

These patterns generalize across many domains: recommendation systems, ad ranking, personalization, anomaly detection, forecasting, and more—as long as your models are exported as `SavedModel`s.

## Best Practices

### 1. Treat the SavedModel as the contract

- Standardize on **`SavedModel` exports** from your training pipelines.
- Freeze preprocessing/postprocessing into the model when possible (e.g., use `tf.Transform` or Keras preprocessing layers) to reduce drift between training and serving.
- Validate that the model signature (`inputs`, `outputs`, dtypes, shapes) matches what clients expect.

### 2. Use clear model and version naming

- Use stable **model names** (e.g., `churn_model`, `ranking_model`) and numeric version directories (`0001`, `0002`, ...).
- Keep model metadata (owner, training run ID, data snapshot) in a registry or alongside the model directory.
- Automate cleanup of old versions to control disk usage.

### 3. Separate training and serving concerns

- Training code should **only export models**; serving concerns (Docker, Kubernetes, load balancing) live elsewhere.
- Use CI/CD or pipelines to move validated models into the **serving model repository**.
- Avoid running heavy training logic inside TF Serving itself—keep it focused on inference.

### 4. Design for horizontal scaling

- Run multiple TF Serving replicas behind a load balancer.
- Ensure servers are **stateless**: no local persistent state beyond ephemeral caches.
- Use readiness/liveness probes (or equivalent) so orchestration systems can safely restart unhealthy pods.

### 5. Right-size batching and hardware

- Start with conservative batch sizes, then benchmark and tune.
- For GPUs, aim for high utilization (70–90%) while staying within latency SLOs.
- Consider separate deployments for **latency-sensitive** vs **throughput-oriented** traffic (e.g., small vs large batch sizes).

### 6. Make observability a first-class concern

- Export and scrape metrics (latency, error rates, model load events) from each TF Serving instance.
- Correlate TF Serving metrics with upstream application logs.
- Capture which **model version** served each request for easier debugging and audits.

## Common Pitfalls

### 1. Misconfigured model paths

**Symptoms**
- Server starts but returns `NOT_FOUND: Servable not found` or `Model not found`.
- Metrics show zero successful requests.

**Causes**
- `--model_base_path` or `models.config` points to the wrong directory.
- Version subdirectories are missing or not numeric (e.g., `v1` instead of `0001`).

**How to avoid**
- Verify the on-disk layout: `models/<model_name>/<version>/saved_model.pb`.
- Log the resolved paths at startup and validate in your deployment manifests.

---

### 2. Input/output signature mismatches

**Symptoms**
- Errors like `Invalid argument: input tensor ... not found` or shape/dtype mismatch errors.

**Causes**
- Client sends JSON with different field names than the SavedModel signature.
- Different preprocessing between training and serving (feature order, normalization, etc.).

**How to avoid**
- Inspect the SavedModel signature with `saved_model_cli show --all --dir <path>`.
- Keep a **single source of truth** for feature ordering and preprocessing (e.g., Keras preprocessing layers).
- Write contract tests that call TF Serving and validate responses.

---

### 3. Cold-start latency

**Symptoms**
- First request after deployment takes seconds longer than subsequent ones.

**Causes**
- Model weights, graphs, and kernels are lazily loaded/optimized on first use.

**How to avoid**
- Use **model warmup** files so TF Serving runs synthetic requests at startup.
- Trigger a warmup request after deployment from a health-check job.

---

### 4. Overloaded servers (high latency / timeouts)

**Symptoms**
- p95/p99 latencies spike under load.
- Upstream services see timeouts or `UNAVAILABLE` errors.

**Causes**
- Too few server replicas for the incoming QPS.
- Batching misconfigured (too large or too small batches).
- Insufficient CPU/GPU resources or noisy neighbors.

**How to avoid**
- Benchmark and tune batch sizes and concurrency.
- Add horizontal replicas and proper autoscaling rules.
- Isolate critical workloads to dedicated nodes/GPUs.

---

### 5. Uncontrolled model version sprawl

**Symptoms**
- Disk fills up; server spends time loading/unloading many versions.

**Causes**
- Training pipeline continuously writes new versions without retention.

**How to avoid**
- Implement lifecycle policies to keep only relevant versions.
- Use `model_version_policy` to restrict which versions are actually loaded.

## Performance Optimization

Optimizing TensorFlow Serving is mostly about **matching concurrency, batching, and hardware** to your traffic patterns.

### 1. Benchmark before tuning

- Start with a **baseline** using a simple load generator (e.g., `ab`, `wrk`, Locust, or a Python script).
- Measure:
  - p50 / p95 / p99 latency
  - Throughput (QPS)
  - CPU / GPU utilization
- Only then adjust flags and deployment parameters.

### 2. Tune batching

- Enable server-side batching for GPU-backed deployments or high-QPS CPU services.
- Key ideas:
  - Larger batches → better hardware utilization but higher tail latency.
  - Smaller batches → lower latency but less throughput.
- Use conservative defaults (e.g., max batch size 8–32, batch timeout 10–50 ms) and iterate.

### 3. Control concurrency and threading

- Use deployment-level knobs (replica count, CPU/GPU resources) plus TF Serving flags to control concurrency.
- For CPU-bound models, ensure you allocate enough cores and tune thread pools.
- For GPU-bound models, avoid **over-subscribing** a single GPU; it’s often better to run fewer, well-utilized replicas.

### 4. Right-size hardware

- Use profiling tools (TensorBoard, `nvidia-smi`, vendor-specific profilers) to understand bottlenecks.
- For small models and low QPS, CPU-only serving may be sufficient and cheaper.
- For deep neural nets and high QPS, prefer GPUs with sufficient memory to hold the model plus batch.

### 5. Optimize the model itself

- Consider **quantization** or pruning (via TensorFlow Model Optimization Toolkit) before exporting.
- Remove unnecessary layers or heavy preprocessing that could be done elsewhere.
- Ensure you’re not shipping debug-only ops or extremely large embeddings if not needed.

### 6. Observe and iterate

- Continuously collect metrics on latency, errors, and resource utilization.
- Re-run benchmarks when:
  - Model architecture changes
  - Traffic patterns change
  - Hardware or deployment topology changes

In the next cell you can plug in your own endpoint URL and run a simple latency benchmark against your TF Serving deployment.

In [ ]:
# Simple latency benchmark for a TensorFlow Serving endpoint

import time
import statistics
import requests
import numpy as np

MODEL_NAME = "demo_mlp"  # adjust if you use a different name
URL = f"http://localhost:8501/v1/models/{MODEL_NAME}:predict"

# Small synthetic dataset
inputs = np.random.rand(8, 4).astype("float32")  # batch of 8
payload = {"instances": inputs.tolist()}

def benchmark(url: str, payload: dict, num_requests: int = 20):
    latencies = []
    for i in range(num_requests):
        start = time.time()
        resp = requests.post(url, json=payload)
        elapsed = time.time() - start
        if not resp.ok:
            print(f"Request {i} failed: {resp.status_code} {resp.text}")
        latencies.append(elapsed)
    
    print(f"Requests: {num_requests}")
    print(f"Mean latency: {statistics.mean(latencies):.4f} s")
    print(f"p50: {statistics.median(latencies):.4f} s")
    print(f"min: {min(latencies):.4f} s, max: {max(latencies):.4f} s")

# Uncomment after your TF Serving container is running
# benchmark(URL, payload, num_requests=50)

## Production Deployment

TensorFlow Serving fits naturally into containerized, orchestrated environments.

### 1. Docker deployment (single instance)

For small environments or local testing, you can run a single container per model:

```bash
MODEL_NAME=demo_mlp

# Assuming ./models/$MODEL_NAME contains versioned SavedModels

docker run -p 8500:8500 -p 8501:8501 \
  --mount type=bind,source=$(pwd)/models,target=/models \
  -e MODEL_NAME=${MODEL_NAME} \
  tensorflow/serving \
  --model_name=${MODEL_NAME} \
  --model_base_path=/models/${MODEL_NAME}
```

- Place this behind a reverse proxy or API gateway if you need authentication, rate limiting, or custom routing.

### 2. Docker Compose / multiple services

Use Docker Compose (or similar) to run multiple TF Serving instances plus a gateway:

```yaml
version: "3.9"
services:
  tf-serving:
    image: tensorflow/serving
    environment:
      MODEL_NAME: demo_mlp
    volumes:
      - ./models:/models
    command: [
      "--model_name=demo_mlp",
      "--model_base_path=/models/demo_mlp"
    ]
    ports:
      - "8501:8501"  # REST

  api-gateway:
    image: your-org/api-gateway:latest
    depends_on:
      - tf-serving
    # Gateway routes /predict to tf-serving:8501
```

### 3. Kubernetes deployment

In Kubernetes, you typically:

- Package TensorFlow Serving as a Deployment.
- Expose it via a Service or Ingress.
- Use **horizontal pod autoscaling** based on CPU, QPS, or custom metrics.

Example (simplified) manifest:

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: tf-serving-demo
spec:
  replicas: 2
  selector:
    matchLabels:
      app: tf-serving-demo
  template:
    metadata:
      labels:
        app: tf-serving-demo
    spec:
      containers:
      - name: tf-serving
        image: tensorflow/serving:latest
        args:
          - "--model_name=demo_mlp"
          - "--model_base_path=/models/demo_mlp"
        ports:
          - containerPort: 8501
        volumeMounts:
          - name: model-volume
            mountPath: /models
        resources:
          requests:
            cpu: "500m"
            memory: "2Gi"
          limits:
            cpu: "2"
            memory: "4Gi"
      volumes:
        - name: model-volume
          persistentVolumeClaim:
            claimName: tf-serving-model-pvc
---
apiVersion: v1
kind: Service
metadata:
  name: tf-serving-demo
spec:
  selector:
    app: tf-serving-demo
  ports:
    - name: http
      port: 80
      targetPort: 8501
  type: LoadBalancer
```

### 4. GPU-enabled deployments

- Use the `tensorflow/serving:latest-gpu` image.
- Request GPUs in your pod spec (`nvidia.com/gpu: 1`).
- Make sure the cluster nodes have the NVIDIA drivers and device plugin installed.

### 5. CI/CD and model rollout

- Link your **training pipeline** to a **deployment pipeline**:
  - When a model passes evaluation, export to the model repository.
  - Update a `ConfigMap` or environment variable that points TF Serving to the new version or path.
  - Roll out pods with a rolling update strategy.
- Keep rollout steps small and observable so you can roll back quickly if metrics degrade.

## Monitoring and Observability

In production, TensorFlow Serving should be treated like any other critical microservice: you need **metrics, logs, and traces**.

### 1. Key metrics to track

At a minimum, track:

- **Request latency**
  - p50 / p95 / p99 for prediction endpoints
  - Broken down by model name and version
- **Error rates**
  - HTTP/gRPC status codes (4xx, 5xx)
  - Application-level errors (invalid inputs, failed model loads)
- **Throughput (QPS)**
  - Requests per second per instance and per cluster
- **Resource utilization**
  - CPU, memory
  - GPU utilization and memory (if applicable)

You can export metrics via:

- **Sidecars** or agents (Prometheus node exporter, OpenTelemetry collectors)
- Application-level metrics in front of TF Serving (API gateway, backend services)

### 2. Logging best practices

- Capture at least:
  - Request metadata (model, version, request size, correlation IDs)
  - Response status and latency
  - Error details (without logging sensitive data)
- Centralize logs in a system like Elasticsearch, Cloud Logging, or Loki.
- Use **structured logs** (JSON) for easier querying and aggregation.

### 3. Tracing

If you use distributed tracing (Jaeger, Zipkin, OpenTelemetry):

- Treat TF Serving as a **downstream dependency** in your traces.
- Propagate trace IDs from the client through your gateway and backend to the calls that reach TF Serving.

### 4. Model-level observability

Beyond infra metrics, it’s useful to observe **model behavior over time**:

- Input feature distributions (to detect data drift).
- Output score distributions (to detect calibration issues).
- Per-model-version metrics so you can correlate incidents with rollouts.

This typically requires logging features/predictions from the caller or a dedicated **model monitoring** component, not TF Serving alone.

## Troubleshooting

### Issue 1: `NOT_FOUND: Servable not found`

**Symptoms**
- REST calls return errors like:
  - `{"error": "Servable (model) not found..."}`
- gRPC calls fail with `NOT_FOUND` status.

**Likely causes**
- `MODEL_NAME` environment variable doesn’t match the name you use in the URL.
- `--model_name` flag is different from the name in the request path.
- `--model_base_path` (or `models.config`) points to the wrong directory.

**How to fix**
- Ensure the URL uses the same name as `--model_name`, for example:
  - `http://localhost:8501/v1/models/demo_mlp:predict`
- Verify the directory layout on disk:
  - `models/demo_mlp/0001/saved_model.pb`
- Check container logs at startup for model load errors.

---

### Issue 2: `InvalidArgument` errors about inputs

**Symptoms**
- Errors like `input tensor ... not found` or shape/type mismatch messages.

**Likely causes**
- Client JSON doesn’t match the SavedModel signature.
- Input feature order or names differ between training and serving.

**How to fix**
- Inspect the SavedModel signature:

  ```bash
  saved_model_cli show --all --dir models/demo_mlp/0001
  ```

- Adjust your request payload to match the expected field names, shapes, and dtypes.
- Consider wrapping preprocessing into the model graph to reduce divergence.

---

### Issue 3: High latency / timeouts under load

**Symptoms**
- p95/p99 latency spikes during traffic spikes.
- Upstream services see request timeouts.

**Likely causes**
- Insufficient replicas for incoming QPS.
- No batching or poorly tuned batching.
- Contention for CPU/GPU resources.

**How to fix**
- Add more TF Serving replicas behind the load balancer.
- Enable and tune **batching** for GPU deployments.
- Use resource requests/limits and node isolation for critical services.

---

### Issue 4: GPU not utilized or CUDA errors

**Symptoms**
- GPU utilization stays at 0%.
- Logs show missing CUDA libraries or device errors.

**Likely causes**
- Using the CPU-only image instead of `tensorflow/serving:latest-gpu`.
- NVIDIA drivers or runtime not properly installed on the host.

**How to fix**
- Switch to the GPU image and ensure the NVIDIA Container Toolkit is configured.
- Verify `nvidia-smi` works on the host and inside a test container.

---

### Issue 5: Disk usage growing without bound

**Symptoms**
- Model storage fills the volume over time.

**Likely causes**
- Training pipeline writes many model versions without retention.

**How to fix**
- Implement a retention policy in your training/deployment pipeline.
- Use `model_version_policy` to limit how many versions TF Serving actually loads.

## Comparison with Alternatives

TensorFlow Serving is one option in a broader ecosystem of model-serving tools.

| Capability / Tool            | TensorFlow Serving            | Triton Inference Server                     | TorchServe                        | BentoML / OpenLLM                          |
|-----------------------------|--------------------------------|---------------------------------------------|-----------------------------------|--------------------------------------------|
| Primary focus               | TensorFlow `SavedModel`s      | Multi-framework (TF, PyTorch, ONNX, etc.)   | PyTorch models                    | General ML/LLM apps                        |
| Implementation language     | C++                           | C++                                         | Java + Python                     | Python (service framework)                 |
| Protocols                   | gRPC, REST                    | gRPC, HTTP/REST                            | HTTP/REST                        | HTTP/REST, gRPC                            |
| Batching                    | Yes (server-side)             | Yes (advanced GPU batching, ensembles)      | Yes                               | Yes (adaptive batching via runners)        |
| Multi-model, multi-version  | Yes                           | Yes                                         | Yes                               | Yes (per-service design)                   |
| GPU support                 | Yes (GPU image)               | Yes (NVIDIA-optimized)                      | Yes                               | Yes (depends on underlying frameworks)     |
| Best fit                    | TF/TFX pipelines, classic ML  | GPU-intensive DL/LLM, multi-framework       | PyTorch-only orgs                | App-level orchestration + flexible serving |

### When to choose TensorFlow Serving

Consider TensorFlow Serving when:

- Your models are primarily **TensorFlow/Keras** and exported as `SavedModel`s.
- You want a **stable, well-established** serving stack with long production history.
- You are already invested in the **TFX ecosystem**.
- You prefer a **simple, focused** component that only does inference, while other systems handle routing and business logic.

Consider alternatives when:

- You need a **multi-framework** GPU-optimized server (Triton, vLLM, TGI).
- You serve mostly **PyTorch** models (TorchServe, Triton, BentoML/OpenLLM).
- You want a **Python-centric application framework** where you embed business logic directly in the serving layer (FastAPI, BentoML, Mosec, etc.).

## Resources

### Official documentation

- **TensorFlow Serving overview**  
  https://www.tensorflow.org/tfx/guide/serving
- **TensorFlow Serving GitHub repository**  
  https://github.com/tensorflow/serving
- **SavedModel format**  
  https://www.tensorflow.org/guide/saved_model

### Tutorials and guides

- **Serving a TensorFlow model** (official tutorial)  
  https://www.tensorflow.org/tfx/guide/serving#serving_a_model
- **TensorFlow Serving with Docker**  
  https://www.tensorflow.org/tfx/guide/serving#docker_images
- Example blog posts and walkthroughs (search terms):
  - "TensorFlow Serving Docker tutorial"
  - "TensorFlow Serving Kubernetes example"
  - "TFX and TensorFlow Serving end-to-end pipeline"

### Community resources

- TensorFlow discussion forums:  
  https://discuss.tensorflow.org/
- Stack Overflow (tag: `tensorflow-serving`):  
  https://stackoverflow.com/questions/tagged/tensorflow-serving
- TensorFlow YouTube channel (talks, demos):  
  https://www.youtube.com/tensorflow

### Related technologies

- **TFX** – end-to-end ML pipelines that often deploy to TensorFlow Serving.  
- **NVIDIA Triton Inference Server** – multi-framework GPU-optimized serving.  
- **TorchServe** – serving for PyTorch models.  
- **BentoML / OpenLLM** – application-focused serving framework for diverse ML models.  
- **KServe (KFServing)** – Kubernetes-native model-serving platform that can use TensorFlow Serving as a backend.